# 📊 Business Analytics & Insights

## Overview
This section focuses on **answering key business questions** using the cleaned and 
structured data prepared in the previous layers.

---

## Business Question We Will Answer

### 🛒 Customer Journey & Conversion
> *Where do users drop off before completing a purchase ?*

Understanding the conversion funnel helps identify friction points in the purchase journey.
By measuring how many users progress from their first visit to a completed transaction,
we can pinpoint exactly where the business is losing potential customers.

### Silver Events — Column Description

| Column | Type | Description |
|---|---|---|
| `user_id` | String | Unique identifier for each user |
| `device` | String | Device used by the user (Windows, macOS, iOS, Android...) |
| `event_name` | String | Type of action performed by the user on the site (main, mattresses, add_item...) |
| `traffic_source` | String | Channel that brought the user to the site (google, facebook, email...) |
| `event_previous_timestamp` | Timestamp | Timestamp of the previous action performed by the user — null if first event |
| `event_timestamp` | Timestamp | Timestamp of the current action |
| `user_first_touch_timestamp` | Timestamp | Timestamp of the very first time the user visited the site |
| `city` | String | City from which the user performed the action |
| `state` | String | US state from which the user performed the action |

## 1. Funnel Conversion Analysis

### Business Question
**How many users convert from page view to purchase ?**

At each stage of the customer journey, some users drop off and never complete the next step.
Understanding where and how many users drop off allows the business to identify
friction points and optimize the user experience to increase revenue.

### The Funnel
```
Stage 1 — page_view      → user lands on the site
     ↓ drop off
Stage 2 — search         → user searches for a product
     ↓ drop off
Stage 3 — product page   → user clicks on a product
     ↓ drop off
Stage 4 — add_to_cart    → user adds a product to the cart
     ↓ drop off
Stage 5 — purchase       → user completes the transaction ✅
```

### What we will measure
- **Total unique users** at each stage
- **Conversion rate** from one stage to the next
- **Overall conversion rate** from first visit to purchase

### Loading event table & display

In [0]:
events = spark.read.table("ecom_clickstream.gold.fact_event")
events.createOrReplaceTempView("fact_event")

In [0]:
%sql
SELECT *
FROM fact_event

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5144283242871527>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT *\nFROM fact_event\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, line, cell)
    186 except BaseException as e:
    187     self.driver_activity_logger.logExecuteCo

### Understanding `event_name` Values

The `event_name` column in `silver.events` tracks every user action on the site.
There are 23 distinct event types grouped into 4 categories :

**Main Navigation**
- `main` → homepage of the site
- `original` → Original mattress collection page
- `premium` → Premium mattress collection page
- `foam` → Foam mattress collection page
- `down` → Down pillow collection page

**Product Catalog**
- `mattresses` → mattress catalog page
- `pillows` → pillow catalog page

**Information & Support**
- `faq` → frequently asked questions page
- `warranty` → product warranty page
- `delivery` → delivery information page
- `reviews` → customer reviews page
- `press` → press & media page
- `careers` → recruitment page

**Purchase Funnel**
- `add_item` → user adds a product to the cart
- `cart` → user views the cart
- `checkout` → user starts the checkout process
- `email_coupon` → user enters a promo code
- `shipping_info` → user fills in shipping information
- `cc_info` → user fills in payment information
- `finalize` → user confirms and completes the order ✅

**User Account**
- `guest` → browsing as a guest (no account)
- `register` → creating a new account
- `login` → logging into an existing account

---

> 💡 For the funnel conversion analysis, we focus on the **Purchase Funnel** events only —
> from `add_item` to `finalize` — as these represent the critical path to conversion.

In [0]:
%sql
SELECT 
    event_name,
    COUNT(DISTINCT user_id) AS unique_users
FROM fact_event
GROUP BY event_name
ORDER BY unique_users DESC

### Funnel Code

In [0]:
%sql
SELECT 
    event_name,
    COUNT(DISTINCT user_id) AS unique_users
FROM fact_event
WHERE event_name IN ('main', 'mattresses', 'add_item', 'cart', 'checkout', 'shipping_info', 'cc_info', 'finalize')
GROUP BY event_name

In [0]:
%sql

  SELECT
    event_name as stage,
    COUNT(DISTINCT user_id) AS unique_user,
    CASE event_name
      WHEN 'main' THEN 1
      WHEN 'add_item' THEN 2
      WHEN 'cart' THEN 3
      WHEN 'checkout' THEN 4
      WHEN 'shipping_info' THEN 5
      WHEN 'cc_info' THEN 6
      WHEN 'finalize' THEN 7
    END AS stage_order
  FROM fact_event
  where event_name IN('main', 'add_item', 'cart', 'checkout', 'shipping_info', 'cc_info', 'finalize')
  GROUP BY event_name
  ORDER BY stage_order


In [0]:
%sql
WITH funnel AS (
      SELECT
    event_name as stage,
    COUNT(DISTINCT user_id) AS unique_users,
    CASE event_name
      WHEN 'main' THEN 1
      WHEN 'add_item' THEN 2
      WHEN 'cart' THEN 3
      WHEN 'checkout' THEN 4
      WHEN 'shipping_info' THEN 5
      WHEN 'cc_info' THEN 6
      WHEN 'finalize' THEN 7
    END AS stage_order
  FROM fact_event
  where event_name IN('main', 'add_item', 'cart', 'checkout', 'shipping_info', 'cc_info', 'finalize')
  GROUP BY event_name
  ORDER BY stage_order
)

SELECT
    stage_order,
    stage,
    unique_users,
    ROUND(unique_users * 100.0 / FIRST_VALUE(unique_users) OVER (ORDER BY stage_order), 2) AS pct_of_top,
    ROUND(unique_users * 100.0 / LAG(unique_users)         OVER (ORDER BY stage_order), 2) AS pct_of_previous
FROM funnel
ORDER BY stage_order

### Understanding the Two Metrics

**`pct_of_top` — % remaining vs the top of the funnel**
Measures how many users are left at each stage compared to the very first step (`main`).
```
main      → 289,460 users → 100%
add_item  →  51,390 users →  17.75%
finalize  →  18,178 users →   6.28%
```
> Out of 100 users who land on the site, only **6.28% complete a purchase**.

**`pct_of_previous` — % remaining vs the previous stage**
Measures how many users move from one stage to the next.

---

### Key Business Insights

**🚨 Biggest drop — `main` → `add_item` (82% abandon)**
The majority of visitors browse the site without ever adding a product to the cart.
- Opportunity : improve product pages, call-to-action buttons, and pricing visibility

**⚠️ Second biggest drop — `checkout` → `shipping_info` (26% abandon)**
A significant portion of users abandon at the shipping information step.
- Opportunity : offer free shipping or display shipping costs earlier in the journey

**⚠️ Third drop — `shipping_info` → `cc_info` (26% abandon)**
Another 26% drop off at the payment information step.
- Opportunity : add more payment options, display security badges, simplify the form

**✅ Strong point — `cart` → `checkout` (99.78% conversion)**
Almost every user who views their cart proceeds to checkout.
- The cart page is well optimized and creates strong purchase intent

In [0]:
%sql
-- Funnel via Mattresses
WITH funnel_mattresses AS (
    SELECT
        event_name AS stage,
        COUNT(DISTINCT user_id) AS unique_users,
        CASE event_name
            WHEN 'main'        THEN 1
            WHEN 'mattresses'  THEN 2
            WHEN 'add_item'    THEN 3
        END AS stage_order
    FROM fact_event
    WHERE event_name IN ('main', 'mattresses', 'add_item')
    GROUP BY event_name
)
SELECT
    stage_order,
    stage,
    unique_users,
    ROUND(unique_users * 100.0 / FIRST_VALUE(unique_users) OVER (ORDER BY stage_order), 2) AS pct_of_top,
    ROUND(unique_users * 100.0 / LAG(unique_users)         OVER (ORDER BY stage_order), 2) AS pct_of_previous
FROM funnel_mattresses
ORDER BY stage_order

In [0]:
%sql
-- Funnel via Pillows
WITH funnel_pillows AS (
    SELECT
        event_name AS stage,
        COUNT(DISTINCT user_id) AS unique_users,
        CASE event_name
            WHEN 'main'     THEN 1
            WHEN 'pillows'  THEN 2
            WHEN 'add_item' THEN 3
        END AS stage_order
    FROM fact_event
    WHERE event_name IN ('main', 'pillows', 'add_item')
    GROUP BY event_name
)
SELECT
    stage_order,
    stage,
    unique_users,
    ROUND(unique_users * 100.0 / FIRST_VALUE(unique_users) OVER (ORDER BY stage_order), 2) AS pct_of_top,
    ROUND(unique_users * 100.0 / LAG(unique_users)         OVER (ORDER BY stage_order), 2) AS pct_of_previous
FROM funnel_pillows
ORDER BY stage_order

### Key Insight — Mattresses vs Pillows Funnel Comparison

| Stage | Mattresses Users | Pillows Users |
|---|---|---|
| main | 289,460 (100%) | 289,460 (100%) |
| category page | 112,730 (38.94%) | 53,103 (18.35%) |
| add_item | 51,390 (17.75%) | 51,390 (17.75%) |

---

**📊 Observation 1 — Mattresses attract far more visitors**
The mattresses category page attracts **2x more visitors** than pillows (38.94% vs 18.35%).
This confirms that mattresses are the primary product driving traffic to the site.

**🎯 Observation 2 — Pillow browsers are much more intentional buyers**
Despite attracting fewer visitors, pillows show a dramatically higher conversion
from category page to `add_item` :
```
mattresses → add_item : 45.59%  — less than half convert
pillows    → add_item : 96.77%  — almost everyone converts
```

Users who visit the pillow page are **highly intentional** — they know what they want
and almost always add a product to the cart.
In contrast, mattress browsers are more likely to be **window shopping** or comparing options.

**💡 Business Recommendation**
- The mattresses category has a **traffic problem converted to intent** —
  invest in better product pages, customer reviews, and comparison tools
  to help undecided mattress browsers commit to a purchase
- The pillows category has a **traffic problem** —
  it converts extremely well but attracts far fewer visitors.
  Invest in cross-selling pillows to mattress browsers
  (e.g. "Complete your bed" section on the mattress page)

> 💡 Both categories end up with the same number of `add_item` users (51,390)
> despite very different journeys — this suggests the site has a natural ceiling
> on purchase intent that is worth investigating further.

### Key Business Insights

**🏆 Top Revenue Driver — Standard Queen Mattress**
The Standard Queen Mattress generates the highest revenue at **$885,951**
despite not being the most ordered product (936 orders vs 961 for Standard Twin).
This is driven by its higher price point compared to the Twin.

**📦 Most Ordered — Standard Twin Mattress**
The Standard Twin Mattress has the highest order count (961) and quantity sold (1,032),
suggesting it is popular for smaller bedrooms, guest rooms, or budget-conscious buyers.
However it ranks 3rd in revenue due to its lower price point.

**🛏️ Mattresses vs Pillows — Revenue Gap**
```
Total mattress revenue → $3,750,637  (99.4% of total revenue)
Total pillow revenue   →    $16,199  (0.6% of total revenue)
```